# Jarvis George — Phase 3: QLoRA Fine-Tuning (v3, refactored)

**Διπλωματική:** Architecting Autonomous Digital Twins · University of Peloponnese
**Μοντέλο:** Mistral-7B-Instruct-v0.2 · QLoRA (r=64, α=128) · 4-bit NF4 · A100

| Στάδιο | Κατάσταση |
|---|---|
| **Part A — Persona-Chat** (109K pairs) | ✅ ΟΛΟΚΛΗΡΩΘΗΚΕ 2026-06-21 — adapters σωσμένα στο Drive |
| **Part B — Viber** (13K pairs, sequential) | ⏳ Εκκρεμεί — απαιτεί `jarvis_training_data_sanitized.json` |

**Τι άλλαξε στο v3:**
1. *Resume-safe*: αν τα Part A adapters υπάρχουν στο Drive, το training του Part A παραλείπεται αυτόματα.
2. *Save στο ίδιο cell με το training* — δεν ξαναχάνουμε adapters επειδή δεν έτρεξε ξεχωριστό save cell.
3. *Sanitization guard*: το Part B **αρνείται** να τρέξει με μη-καθαρισμένα δεδομένα (GDPR).
4. Όλες οι ρυθμίσεις σε ΕΝΑ config cell (καθρέφτης του `config/settings.yaml` στο repo).
5. Three-stage evaluation ενσωματωμένο (baseline → Persona-Chat → Viber).

⚠️ **Runtime → A100 GPU πριν τρέξεις οτιδήποτε** (αλλαγή runtime αργότερα = χάνονται όλα τα πακέτα/variables — γι' αυτό μένουμε σε A100 από την αρχή).

## Cell 1 — Έλεγχος GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Δεν βρέθηκε GPU — Runtime ▸ Change runtime type ▸ A100 GPU")

props = torch.cuda.get_device_properties(0)
# PyTorch μετονόμασε το total_mem → total_memory· το getattr δουλεύει και στα δύο (fix v2)
vram_gb = (getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)) / 1024**3
print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB")

BATCH_SIZE = 4 if vram_gb > 30 else 1        # A100 → 4, T4 fallback → 1
print(f"BATCH_SIZE = {BATCH_SIZE}")

## Cell 2 — Εγκατάσταση πακέτων
*(τρέξε το ΞΑΝΑ μετά από κάθε runtime restart — τα πακέτα χάνονται, γνωστό θέμα v2)*

In [ ]:
%pip install -q "transformers>=4.44,<5" "trl==0.9.6" "peft>=0.11" \
    "bitsandbytes>=0.43" "accelerate>=0.30" "datasets>=2.19"

# flash-attn ΠΡΟΑΙΡΕΤΙΚΟ (15-20 λεπτά build) — το notebook δουλεύει και χωρίς αυτό

## Cell 3 — Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Cell 4 — Config (μοναδικό σημείο ρυθμίσεων — καθρέφτης του config/settings.yaml)

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

CFG = {
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    # LoRA — ΙΔΙΑ με το ολοκληρωμένο Part A run (μην τα αλλάξεις: συμβατότητα adapters)
    "lora_r": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    # Part A — Persona-Chat (ολοκληρωμένο)
    "persona_data": DRIVE / "persona_chat_training_data.json",
    "persona_adapters": DRIVE / "jarvis_models/persona_chat_qlora",
    "persona_lr": 2e-4,
    # Part B — Viber (sequential, χαμηλότερο LR)
    "viber_sanitized_data": DRIVE / "jarvis_training_data_sanitized.json",
    "viber_raw_data": DRIVE / "jarvis_training_data.json",   # ΜΟΝΟ για τον guard έλεγχο
    "viber_adapters": DRIVE / "jarvis_models/viber_qlora",
    "viber_lr": 1e-4,
    # κοινά
    "max_seq_length": 512,
    "num_epochs": 1,
    "save_steps": 50,            # μάθημα v2: ήταν 500 και χάθηκαν 83 steps σε crash
    "eval_out": DRIVE / "jarvis_models/three_stage_eval.json",
}

# Το format είναι byte-identical με το Part A run (Mistral instruct format)
def format_pair(instruction: str, response: str) -> str:
    return f"<s>[INST] {instruction.strip()} [/INST] {response.strip()}</s>"

print("Config OK")

---
# Part A — Persona-Chat (✅ ολοκληρωμένο — resume-safe)

Το Part A ολοκληρώθηκε στις 2026-06-21 (648/648 steps, loss 0.446→0.371, ~17 units).
Το επόμενο cell ελέγχει αν τα adapters υπάρχουν στο Drive: αν ναι, **παραλείπει** το
training και φορτώνει τα έτοιμα. Ξανατρέχεις Part A μόνο αν θέλεις αναπαραγωγή από το μηδέν.

## Cell 5 — Έλεγχος υπαρχόντων adapters

In [ ]:
adapter_file = CFG["persona_adapters"] / "adapter_model.safetensors"
SKIP_PART_A = adapter_file.exists()

if SKIP_PART_A:
    size_mb = adapter_file.stat().st_size / 1024**2
    print(f"✅ Βρέθηκαν Part A adapters ({size_mb:.0f} MB) — το training θα ΠΑΡΑΛΕΙΦΘΕΙ.")
else:
    print("⚠️ Δεν βρέθηκαν adapters — το Part A θα εκτελεστεί κανονικά (~1h40m σε A100).")

## Cell 6 — Φόρτωση base μοντέλου (4-bit NF4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_kwargs = dict(quantization_config=bnb_config, device_map="auto")
try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("flash-attn: ενεργό")
except ImportError:
    print("flash-attn: μη διαθέσιμο — συνεχίζουμε κανονικά (fix v2)")

tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"])
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(CFG["base_model"], **model_kwargs)
model.config.use_cache = False
print("Base model φορτώθηκε (~4GB σε 4-bit)")

## Cell 7 — Baseline generation helper + πρώτο στάδιο evaluation

In [ ]:
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.7, do_sample=True,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Σταθερά probes (ίδια με src/jarvis/evaluation/three_stage.py — ΜΗΝ τα αλλάξεις μεταξύ σταδίων)
PROBES = [
    "Τι κάνεις; Όλα καλά;",
    "Θα έρθεις τελικά το Σάββατο;",
    "Μπορείς να μου στείλεις την αναφορά μέχρι αύριο;",
    "Πώς σου φάνηκε η συνάντηση σήμερα;",
    "Έχεις κανένα νέο για το project;",
    "Τι λες να φάμε το βράδυ;",
    "Can you join the call at 3pm tomorrow?",
    "Ευχαριστώ πολύ για τη βοήθεια χθες!",
    "Πότε μπορούμε να τα πούμε από κοντά;",
    "Στείλε μου όταν φτάσεις σπίτι.",
]

STAGE_OUTPUTS = {}
STAGE_OUTPUTS["1_baseline"] = [generate(p) for p in PROBES]
print("Στάδιο 1 (baseline) καταγράφηκε.")
for p, o in zip(PROBES[:3], STAGE_OUTPUTS["1_baseline"][:3]):
    print(f"\nQ: {p}\nA: {o[:200]}")

## Cell 8 — Part A training (τρέχει ΜΟΝΟ αν δεν υπάρχουν adapters)
**Το save γίνεται στο ΙΔΙΟ cell** — μάθημα από το χαμένο Cell 11 του v2.

In [ ]:
import json
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer

if SKIP_PART_A:
    print("SKIP — τα adapters υπάρχουν ήδη· πήγαινε στο Cell 9.")
else:
    records = json.loads(CFG["persona_data"].read_text(encoding="utf-8"))
    texts = [format_pair(r["instruction"], r["response"]) for r in records
             if r.get("instruction") and r.get("response")]
    print(f"Persona-Chat pairs: {len(texts):,}")
    dataset = Dataset.from_dict({"text": texts})

    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(
        r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
        target_modules=CFG["target_modules"], bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    args = SFTConfig(
        output_dir="/content/checkpoints_persona",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=42,          # ≈648 optimizer steps στο πλήρες dataset
        learning_rate=CFG["persona_lr"],
        num_train_epochs=CFG["num_epochs"],
        save_steps=CFG["save_steps"],            # checkpoint κάθε 50 (fix v2)
        logging_steps=10,
        bf16=True,
        max_seq_length=CFG["max_seq_length"],
        dataset_text_field="text",
        report_to="none",
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, tokenizer=tokenizer)
    trainer.train()

    # --- SAVE ΑΜΕΣΩΣ, ΣΤΟ ΙΔΙΟ CELL ---
    CFG["persona_adapters"].mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(CFG["persona_adapters"])
    tokenizer.save_pretrained(CFG["persona_adapters"])
    saved = (CFG["persona_adapters"] / "adapter_model.safetensors").exists()
    print(f"✅ Adapters σώθηκαν στο Drive: {saved}")
    assert saved, "ΤΑ ADAPTERS ΔΕΝ ΣΩΘΗΚΑΝ — μην κλείσεις το runtime!"

## Cell 9 — Φόρτωση Part A adapters + στάδιο 2 evaluation

In [ ]:
from peft import PeftModel

if SKIP_PART_A:
    model = PeftModel.from_pretrained(model, str(CFG["persona_adapters"]), is_trainable=True)
    print("Part A adapters φορτώθηκαν πάνω στο base (trainable — έτοιμοι για Part B).")
else:
    print("Το μοντέλο έχει ήδη τα adapters από το training του Cell 8.")

STAGE_OUTPUTS["2_persona_chat"] = [generate(p) for p in PROBES]
print("Στάδιο 2 (persona-chat) καταγράφηκε.")

---
# Part B — Viber sequential fine-tuning (⏳ εκκρεμεί)

**Sequential = συνεχίζουμε από το ΙΔΙΟ μοντέλο** (validated από PersonaGPT):
καμία επαναφόρτωση fresh model (το bug του v1) — μόνο χαμηλότερο learning rate
(1e-4 αντί 2e-4) ώστε η προσαρμογή στο προσωπικό στυλ να μην σβήσει τα γενικά
conversational patterns του Part A.

## Cell 10 — 🛡️ SANITIZATION GUARD (GDPR)
Το Part B **δεν τρέχει** χωρίς το sanitized αρχείο. Δες `databricks/05_gold_sanitized_export.py` ή `scripts/run_sanitization.py` στο repo.

In [ ]:
import json, re

sanitized_path = CFG["viber_sanitized_data"]

if not sanitized_path.exists():
    raise FileNotFoundError(
        f"Δεν βρέθηκε {sanitized_path}.\n"
        "Τρέξε πρώτα το sanitization (repo: scripts/run_sanitization.py ή databricks/05) "
        "και ανέβασε το jarvis_training_data_sanitized.json στο MyDrive.\n"
        "ΠΟΤΕ training με το raw jarvis_training_data.json — GDPR + memorisation risk."
    )

viber_records = json.loads(sanitized_path.read_text(encoding="utf-8"))
print(f"Sanitized Viber pairs: {len(viber_records):,}")

# Spot-scan: δομικά PII patterns δεν πρέπει να υπάρχουν στο καθαρισμένο αρχείο
blob = json.dumps(viber_records, ensure_ascii=False)
checks = {
    "greek_mobile": r"(?<!\d)(?:\+30\s?)?69\d{8}(?!\d)",
    "iban": r"\bGR\d{25}\b",
    "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
}
hits = {name: len(re.findall(pat, blob)) for name, pat in checks.items()}
print("Residual PII scan:", hits)
assert not any(hits.values()), f"ΒΡΕΘΗΚΕ PII στο 'sanitized' αρχείο — ΣΤΑΜΑΤΑ: {hits}"
print("✅ Guard πέρασε — ασφαλές για training.")

## Cell 11 — Part B training + άμεσο save (ΙΔΙΟ cell)

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

viber_texts = [format_pair(r["instruction"], r["response"]) for r in viber_records
               if r.get("instruction") and r.get("response")]
viber_dataset = Dataset.from_dict({"text": viber_texts})
print(f"Training pairs: {len(viber_texts):,}")

args_b = SFTConfig(
    output_dir="/content/checkpoints_viber",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=8,
    learning_rate=CFG["viber_lr"],               # 1e-4 — χαμηλότερο, sequential
    num_train_epochs=2,                          # μικρό dataset → 2 epochs
    save_steps=CFG["save_steps"],
    logging_steps=10,
    bf16=True,
    max_seq_length=CFG["max_seq_length"],
    dataset_text_field="text",
    report_to="none",
)
trainer_b = SFTTrainer(model=model, args=args_b, train_dataset=viber_dataset, tokenizer=tokenizer)
trainer_b.train()

# --- SAVE ΑΜΕΣΩΣ ---
CFG["viber_adapters"].mkdir(parents=True, exist_ok=True)
trainer_b.model.save_pretrained(CFG["viber_adapters"])
tokenizer.save_pretrained(CFG["viber_adapters"])
saved = (CFG["viber_adapters"] / "adapter_model.safetensors").exists()
print(f"✅ Viber adapters σώθηκαν στο Drive: {saved}")
assert saved, "ΤΑ ADAPTERS ΔΕΝ ΣΩΘΗΚΑΝ — μην κλείσεις το runtime!"

## Cell 12 — Στάδιο 3 evaluation + αποθήκευση σύγκρισης

In [ ]:
import json
from datetime import datetime, timezone

STAGE_OUTPUTS["3_viber"] = [generate(p) for p in PROBES]

result = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "probes": PROBES,
    "stages": STAGE_OUTPUTS,
}
CFG["eval_out"].parent.mkdir(parents=True, exist_ok=True)
CFG["eval_out"].write_text(json.dumps(result, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Σώθηκε: {CFG['eval_out']}")

# Side-by-side προεπισκόπηση
for i, probe in enumerate(PROBES):
    print(f"\n{'='*80}\nQ{i+1}: {probe}")
    for stage, outs in STAGE_OUTPUTS.items():
        print(f"  [{stage}] {outs[i][:160]}")

## Cell 13 — Playground 🎮
Δοκίμασε ελεύθερα το digital twin σου.

In [ ]:
while True:
    try:
        message = input("Εσύ: ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not message or message.lower() in {"exit", "quit"}:
        break
    print(f"Jarvis: {generate(message)}\n")

---
# Επόμενα βήματα

1. ✅ Adapters (Part A + B) στο Drive → κράτα και τοπικό backup.
2. Κατέβασε το `three_stage_eval.json` → πίνακας σύγκρισης για τη διπλωματική (κεφ. Evaluation).
3. Phase 4: `src/jarvis/inference/api.py` (FastAPI) με `JARVIS_ADAPTERS=<viber_qlora>`.
4. Επόμενες μετρήσεις: NLI faithfulness + persona authentication (δες `docs/architecture.md`).